# DeepSeek LLM 7B Chat — Prompting Techniques for Reasoning

Comparing **zero-shot**, **few-shot**, **few-shot CoT**, and **zero-shot CoT** prompting on the `facebook/natural_reasoning` dataset.

**Model**: `deepseek-ai/deepseek-llm-7b-chat` (4-bit quantized)

In [1]:
!pip install transformers accelerate bitsandbytes
!pip install rouge-score bert-score tqdm

In [2]:
import json
import re
import time
import torch
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from rouge_score import rouge_scorer
from bert_score import score as compute_bert_score
from huggingface_hub import login
from google.colab import drive, userdata, files

In [3]:
login(token=userdata.get("HF_TOKEN"))
print("Logged into HuggingFace")

Logged into HuggingFace


## Configuration

In [4]:
MODEL_NAME = "deepseek-ai/deepseek-llm-7b-chat"
DRIVE_DATA_PATH = "/content/drive/MyDrive/NLP Project/Data"
SEED = 42
ANSWER_MARKER = "ANSWER:"

TECHNIQUES = ["zero_shot", "few_shot", "few_shot_cot", "zero_shot_cot"]

MAX_TOKENS_BY_TYPE = {
    "single_word": 256,
    "short": 512,
    "long": 1024,
    "no_answer": 1024,
}

In [5]:
def get_batch_size(avg_input_tokens):
    if avg_input_tokens < 500:
        return 4
    elif avg_input_tokens < 1000:
        return 3
    elif avg_input_tokens < 1500:
        return 2
    else:
        return 1

## Dataset

In [6]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
with open(f'{DRIVE_DATA_PATH}/sampled.jsonl', 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

ref_70b = {}
for item in dataset:
    if item.get("responses"):
        ref_70b[item["sample_id"]] = item["responses"][0]["response"]

print(f"Loaded {len(dataset)} questions")
print(f"70B reference responses available: {len(ref_70b)}")

Loaded 100 questions
70B reference responses available: 100


In [8]:
df_data = pd.DataFrame(dataset)
print(df_data['answer_type'].value_counts())
print()
df_data[['sample_id', 'answer_type', 'question']].head(10)

answer_type
short          37
long           35
no_answer      18
single_word    10
Name: count, dtype: int64



,sample_id,answer_type,question
0,0,short,Given the information about Company B and Comp...
1,1,no_answer,Prove or disprove that an upper bound for $\rh...
2,2,single_word,A block of mass $m$ is released from a height ...
3,3,long,Discuss the role of dopamine in the developmen...
4,4,long,Construct a voltmeter with max impact/effect a...
5,5,short,"Prove that if a^2 is algebraic over a field F,..."
6,6,long,"Compare and contrast the concepts of Belief, T..."
7,7,short,Given an affine variety $X \subset \mathbb{C}^...
8,8,long,Explain how the property of spin in subatomic ...
9,9,single_word,"Given $x=20062007$, and let $$A=\sqrt{x^2+\sqr..."


## Model

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

Model loaded: deepseek-ai/deepseek-llm-7b-chat
GPU memory used: 5.11 GB
GPU memory reserved: 5.25 GB


## Prompting Techniques

In [10]:
FEW_SHOT_EXAMPLES = [
    {
        "question": "A projectile is launched at an angle of 30° above the horizontal with an initial speed of 50 m/s. Calculate the maximum height reached by the projectile. Use g = 9.8 m/s².",
        "answer": "The maximum height is approximately 31.9 meters.",
        "reasoning": "The vertical component of the initial velocity is v₀y = 50 × sin(30°) = 50 × 0.5 = 25 m/s. At maximum height, the vertical velocity becomes zero. Using the kinematic equation v² = v₀² - 2gh, we set v = 0: 0 = (25)² - 2(9.8)h, which gives 0 = 625 - 19.6h. Solving: h = 625 / 19.6 ≈ 31.9 m."
    },
    {
        "question": "Prove that the sum of the first n odd numbers equals n².",
        "answer": "By mathematical induction, the sum 1 + 3 + 5 + ... + (2n-1) = n².",
        "reasoning": "We prove by induction. Base case: n=1, the first odd number is 1, and 1² = 1. ✓ Inductive step: Assume 1 + 3 + ... + (2k-1) = k². Adding the next odd number (2k+1): k² + (2k+1) = k² + 2k + 1 = (k+1)². ✓ By induction, the statement holds for all positive integers n."
    },
    {
        "question": "Explain why metals are good conductors of electricity compared to non-metals.",
        "answer": "Metals conduct electricity well because metallic bonding creates delocalized electrons that are free to move throughout the material when a voltage is applied.",
        "reasoning": "In metallic bonding, atoms release their outer valence electrons into a shared 'electron sea' that permeates the crystal lattice. These delocalized electrons are not bound to any particular atom and can move freely. When an electric field is applied, these free electrons drift in the direction of the field, creating an electric current. Non-metals have their electrons tightly bound in covalent bonds or localized orbitals, leaving very few free charge carriers available to conduct electricity."
    }
]

In [11]:
def build_messages(question, technique):
    if technique == "zero_shot":
        return [
            {"role": "system", "content": "You are a knowledgeable assistant. Answer the following question. After your explanation, state your final answer on a new line starting with 'ANSWER:'"},
            {"role": "user", "content": question}
        ]

    elif technique == "few_shot":
        messages = [
            {"role": "system", "content": "You are a knowledgeable assistant. Answer questions thoroughly. State your final answer on a new line starting with 'ANSWER:'"}
        ]
        for ex in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": f"ANSWER: {ex['answer']}"})
        messages.append({"role": "user", "content": question})
        return messages

    elif technique == "few_shot_cot":
        messages = [
            {"role": "system", "content": "You are a knowledgeable assistant. Think through problems step by step. State your final answer on a new line starting with 'ANSWER:'"}
        ]
        for ex in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": ex["question"]})
            messages.append({"role": "assistant", "content": f"{ex['reasoning']}\n\nANSWER: {ex['answer']}"})
        messages.append({"role": "user", "content": question})
        return messages

    elif technique == "zero_shot_cot":
        return [
            {"role": "system", "content": "You are a knowledgeable assistant."},
            {"role": "user", "content": f"{question}\n\nLet's approach this systematically. "
            "Identify what is given, determine what needs to be found, and apply "
            "the relevant principles step by step. After your reasoning, state "
            "your final answer on a new line starting with 'ANSWER:'"}
        ]

## Inference

In [12]:
def generate_batch(messages_list, max_new_tokens=1024):
    templated = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_list
    ]
    inputs = tokenizer(
        templated, return_tensors="pt", padding=True,
        truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )

    responses = []
    for i, output in enumerate(outputs):
        prompt_len = inputs["input_ids"][i].shape[0]
        decoded = tokenizer.decode(output[prompt_len:], skip_special_tokens=True)
        responses.append(decoded.strip())
    return responses

## Answer Extraction & Normalization

In [13]:
def normalize_latex(text):
    text = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', text)
    text = re.sub(r'\\boxed\{([^}]+)\}', r'\1', text)
    text = re.sub(r'\$+', '', text)
    text = re.sub(r'\\(left|right|quad|qquad|text|mathrm|mathbf|displaystyle)\b', '', text)
    text = re.sub(r'\\{2,}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_response(response):
    response = re.sub(r'^#+\s*(Step\s*\d+|Solution|Answer|Explanation)[:\s]*', '', response, flags=re.MULTILINE)
    response = re.sub(r"^(I'd be happy to help!?|Sure!?|Here's? (?:the|my) (?:answer|solution|response)[:\s]*)", '', response, flags=re.IGNORECASE)
    return response.strip()

def extract_answer(response):
    if ANSWER_MARKER in response:
        return response.split(ANSWER_MARKER)[-1].strip()

    boxed = re.findall(r'\\boxed\{([^}]+)\}', response)
    if boxed:
        return boxed[-1]

    patterns = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is[:\s]+(.+?)(?:\.|$)',
        r'therefore[,:\s]+(.+?)(?:\.|$)',
    ]
    for pattern in patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()

    sentences = [s.strip() for s in response.rstrip('.').split('.') if s.strip()]
    return sentences[-1] if sentences else response

def normalize(text):
    text = normalize_latex(text)
    return text.lower().strip().rstrip('.')

## Evaluation

In [14]:
rouge_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def exact_match(prediction, reference):
    return normalize(extract_answer(prediction)) == normalize(reference)

def contains_match(prediction, reference):
    ref_norm = normalize(reference)
    pred_norm = normalize(prediction)
    return ref_norm in pred_norm

def compute_rouge_l(prediction, reference):
    return rouge_l_scorer.score(reference, prediction)['rougeL'].fmeasure

def compute_f1_token(prediction, reference):
    pred_tokens = normalize(extract_answer(prediction)).split()
    ref_tokens = normalize(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    common = sum((pred_counts & ref_counts).values())
    if not common:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## Run & Evaluate

In [15]:
def evaluate_technique(df_t):
    technique = df_t["technique"].iloc[0]
    has_ref = df_t[df_t["answer_type"] != "no_answer"].copy()

    has_ref["extracted_answer"] = has_ref["response"].apply(extract_answer)
    has_ref["exact_match"] = has_ref.apply(
        lambda r: exact_match(r["response"], r["reference_answer"]), axis=1
    )
    has_ref["contains_match"] = has_ref.apply(
        lambda r: contains_match(r["response"], r["reference_answer"]), axis=1
    )
    has_ref["rouge_l"] = has_ref.apply(
        lambda r: compute_rouge_l(r["response"], r["reference_answer"]), axis=1
    )
    has_ref["f1_token"] = has_ref.apply(
        lambda r: compute_f1_token(r["response"], r["reference_answer"]), axis=1
    )

    has_ref["ref_70b"] = has_ref["sample_id"].map(ref_70b)
    mask_70b = has_ref["ref_70b"].notna()

    preds_ref = has_ref["response"].tolist()
    refs_ref = has_ref["reference_answer"].tolist()
    preds_70b = has_ref.loc[mask_70b, "response"].tolist()
    refs_70b_list = has_ref.loc[mask_70b, "ref_70b"].tolist()

    all_preds = preds_ref + preds_70b
    all_refs = refs_ref + refs_70b_list
    print(f"Computing BERTScore for {len(all_preds)} pairs...")
    P, R, F1 = compute_bert_score(all_preds, all_refs, lang="en", verbose=False)

    n_ref = len(preds_ref)
    has_ref["bertscore_f1"] = F1[:n_ref].numpy()
    has_ref.loc[mask_70b, "bertscore_vs_70b"] = F1[n_ref:].numpy()
    has_ref.loc[mask_70b, "rouge_l_vs_70b"] = has_ref[mask_70b].apply(
        lambda r: compute_rouge_l(r["response"], r["ref_70b"]), axis=1
    )

    return pd.DataFrame([{
        "technique": technique,
        "exact_match": has_ref["exact_match"].mean(),
        "contains_match": has_ref["contains_match"].mean(),
        "rouge_l": has_ref["rouge_l"].mean(),
        "bertscore_f1": has_ref["bertscore_f1"].mean(),
        "f1_token": has_ref["f1_token"].mean(),
        "rouge_l_vs_70b": has_ref["rouge_l_vs_70b"].mean(),
        "bertscore_vs_70b": has_ref["bertscore_vs_70b"].mean(),
        "avg_response_len": has_ref["response_length"].mean(),
        "avg_gen_time": has_ref["generation_time"].mean(),
    }]).round(4)

In [16]:
def run_technique(technique):
    torch.cuda.empty_cache()
    print(f"\n{'='*60}")
    print(f"Running: {technique}")
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"{'='*60}")

    all_messages = [build_messages(item["question"], technique) for item in dataset]
    templated_for_len = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in all_messages
    ]
    input_lengths = [len(tokenizer.encode(t)) for t in templated_for_len]

    sorted_indices = sorted(range(len(dataset)), key=lambda i: input_lengths[i])
    all_messages = [all_messages[i] for i in sorted_indices]
    sorted_items = [dataset[i] for i in sorted_indices]
    sorted_lengths = [input_lengths[i] for i in sorted_indices]

    avg_len = sum(sorted_lengths) / len(sorted_lengths)
    batch_size = get_batch_size(avg_len)
    print(f"Avg input tokens: {avg_len:.0f} -> batch size: {batch_size}")

    results = []
    for batch_start in tqdm(range(0, len(dataset), batch_size), desc=technique):
        batch_end = min(batch_start + batch_size, len(dataset))
        batch_messages = all_messages[batch_start:batch_end]
        batch_items = sorted_items[batch_start:batch_end]

        batch_max_tokens = max(
            MAX_TOKENS_BY_TYPE.get(item["answer_type"], 1024)
            for item in batch_items
        )

        start = time.time()
        responses = generate_batch(batch_messages, max_new_tokens=batch_max_tokens)
        gen_time = (time.time() - start) / len(responses)

        for item, response in zip(batch_items, responses):
            results.append({
                "sample_id": item["sample_id"],
                "question": item["question"],
                "reference_answer": item["reference_answer"],
                "answer_type": item["answer_type"],
                "technique": technique,
                "response": response,
                "generation_time": gen_time,
                "response_length": len(response.split()),
            })

    df_t = pd.DataFrame(results)
    print(f"Done: {len(df_t)} responses for {technique}")

    out_path = f"/content/results_deepseek_{technique}.json"
    df_t.to_json(out_path, orient="records", indent=2)
    files.download(out_path)

    return evaluate_technique(df_t)

### zero_shot

In [ ]:
metrics_zero_shot = run_technique("zero_shot")
metrics_zero_shot


Running: zero_shot
GPU memory: 5.11 GB
Avg input tokens: 121 -> batch size: 4


zero_shot:   0%|          | 0/25 [00:00<?, ?it/s]

Done: 100 responses for zero_shot


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,technique,exact_match,contains_match,rouge_l,bertscore_f1,f1_token,rouge_l_vs_70b,bertscore_vs_70b,avg_response_len,avg_gen_time
0,zero_shot,0.0,0.0366,0.065,0.7152,0.0,0.2097,0.7256,1.0,70.1627


### few_shot

In [ ]:
metrics_few_shot = run_technique("few_shot")
metrics_few_shot


Running: few_shot
GPU memory: 5.12 GB
Avg input tokens: 306 -> batch size: 4


few_shot:   0%|          | 0/25 [00:00<?, ?it/s]

Done: 100 responses for few_shot


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Computing BERTScore for 164 pairs...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,technique,exact_match,contains_match,rouge_l,bertscore_f1,f1_token,rouge_l_vs_70b,bertscore_vs_70b,avg_response_len,avg_gen_time
0,few_shot,0.0,0.0244,0.058,0.7432,0.0,0.1177,0.7456,1.0,70.2336


### few_shot_cot

In [18]:
metrics_few_shot_cot = run_technique("few_shot_cot")
metrics_few_shot_cot


Running: few_shot_cot
GPU memory: 5.11 GB
Avg input tokens: 621 -> batch size: 3


few_shot_cot:   0%|          | 0/34 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Done: 100 responses for few_shot_cot


<IPython.core.display.Javascript object>

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,technique,exact_match,contains_match,rouge_l,bertscore_f1,f1_token,rouge_l_vs_70b,bertscore_vs_70b,avg_response_len,avg_gen_time
0,few_shot_cot,0.0,0.0244,0.0628,0.7561,0.0,0.0994,0.7602,1.0,91.2948


### zero_shot_cot

In [18]:
metrics_zero_shot_cot = run_technique("zero_shot_cot")
metrics_zero_shot_cot


Running: zero_shot_cot
GPU memory: 5.65 GB
Avg input tokens: 149 -> batch size: 4


zero_shot_cot:   0%|          | 0/25 [00:00<?, ?it/s]

Done: 100 responses for zero_shot_cot


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Computing BERTScore for 164 pairs...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,technique,exact_match,contains_match,rouge_l,bertscore_f1,f1_token,rouge_l_vs_70b,bertscore_vs_70b,avg_response_len,avg_gen_time
0,zero_shot_cot,0.0,0.0244,0.0533,0.7138,0.0,0.2092,0.7248,1.0,85.802


## Results

| Technique | Exact Match | Contains Match | ROUGE-L | BERTScore F1 | F1 Token | ROUGE-L vs 70B | BERTScore vs 70B | Avg Resp Len | Avg Gen Time (s) |
|---|---|---|---|---|---|---|---|---|---|
| Zero-Shot | 0.0 | 0.0366 | 0.065 | 0.7152 | 0.0 | 0.2097 | 0.7256 | 1.0 | 70.1627 |
| Few-Shot | 0.0 | 0.0244 | 0.058 | 0.7432 | 0.0 | 0.1177 | 0.7456 | 1.0 | 70.2336 |
| Few-Shot CoT | 0.0 | 0.0244 | 0.0628 | 0.7561 | 0.0 | 0.0994 | 0.7602 | 1.0 | 91.2948 |
| Zero-Shot CoT | 0.0 | 0.0244 | 0.0533 | 0.7138 | 0.0 | 0.2092 | 0.7248 | 1.0 | 85.802 |
